In [3]:

import urllib.request, urllib.parse, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"
BASE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/production"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

# 1. Sprawdź czy kategoria istnieje
cat = q('*[_type=="category" && slug.current == "gladzie-gipsowe-w-proszku"][0]{ _id, name, "slug": slug.current, "parentSlug": parent->slug.current }')
print("Kategoria:", cat)

# 2. Sprawdź ile produktów ma tę kategorię (po _id)
if cat:
    cat_id = cat["_id"]
    count = q(f'count(*[_type=="product" && category._ref == "{cat_id}"])')
    print(f"Produkty z category._ref == {cat_id}: {count}")

# 3. Sprawdź przez slug
count_slug = q('count(*[_type=="product" && category->slug.current == "gladzie-gipsowe-w-proszku"])')
print(f"Produkty przez slug: {count_slug}")

# 4. Sprawdź przez SLUGS ARRAY (jak frontend)
count_arr = q('count(*[_type=="product" && category->slug.current in ["gladzie-gipsowe-w-proszku"]])')
print(f"Produkty przez in []: {count_arr}")

# 5. Sprawdź przykładowy produkt z tej kategorii
ex = q('*[_type=="product" && category->slug.current == "gladzie-gipsowe-w-proszku"][0]{ name, "slug": slug.current, "catSlug": category->slug.current }')
print("Przykład:", ex)


Kategoria: {'_id': 'cat-l3-gladzie-gipsowe-w-proszku', 'name': 'Gładzie gipsowe w proszku', 'parentSlug': 'gipsy-i-gladzie', 'slug': 'gladzie-gipsowe-w-proszku'}


Produkty z category._ref == cat-l3-gladzie-gipsowe-w-proszku: 1


Produkty przez slug: 1


Produkty przez in []: 1


Przykład: {'catSlug': 'gladzie-gipsowe-w-proszku', 'name': 'Gladz Gipsowa Knauf G K Finish 25 Kg', 'slug': 'gladz-gipsowa-knauf-g-k-finish-25-kg'}


In [7]:

import urllib.request, urllib.parse, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"
BASE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/production"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

# Pobierz całe drzewo kategorii gipsy-i-gladzie
cat = q('''*[_type=="category" && slug.current=="gipsy-i-gladzie"][0]{
  _id, name, "slug": slug.current,
  "children": *[_type=="category" && parent._ref==^._id]{
    _id, name, "slug": slug.current,
    "count": count(*[_type=="product" && category._ref==^._id && !(name match "P-*")])
  }
}''')
print("Kategoria nadrzędna:", cat.get("name"))
total = 0
for ch in cat.get("children", []):
    cnt = ch["count"]
    total += cnt
    print(f"  {ch['name']} ({ch['slug']}): {cnt} produktów")
print(f"RAZEM w podkategoriach: {total}")

# Produkty przez parent chain (jak nowe zapytanie)
c = q('count(*[_type=="product" && (category->slug.current=="gipsy-i-gladzie" || category->parent->slug.current=="gipsy-i-gladzie") && !(name match "P-*")])')
print(f"\nNowe zapytanie (parent chain): {c} produktów")


Kategoria nadrzędna: Gipsy i gładzie
  Gipsy budowlane (gipsy-budowlane): 0 produktów
  Gipsy szpachlowe (gipsy-szpachlowe): 0 produktów
  Gładzie gipsowe w proszku (gladzie-gipsowe-w-proszku): 1 produktów
  Gładzie masy gotowe (gladzie-masy-gotowe): 0 produktów
  Kleje gipsowe (kleje-gipsowe): 0 produktów
RAZEM w podkategoriach: 1



Nowe zapytanie (parent chain): 1 produktów


In [11]:

import urllib.request, urllib.parse, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"
BASE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/production"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())["result"]

# Top-level kategorie z liczbą produktów w całym drzewie
cats = q('''*[_type=="category" && !defined(parent)] | order(name asc) {
  _id, name, "slug": slug.current,
  "directCount": count(*[_type=="product" && category._ref==^._id && !(name match "P-*")]),
  "l2Count": count(*[_type=="product" && category->parent._ref==^._id && !(name match "P-*")]),
  "l3Count": count(*[_type=="product" && category->parent->parent._ref==^._id && !(name match "P-*")])
}''')

total = 0
print(f"{'Kategoria':<30} {'L1':>5} {'L2':>5} {'L3':>5} {'SUM':>6}")
print("-"*55)
for c in cats:
    s = c['directCount'] + c['l2Count'] + c['l3Count']
    total += s
    print(f"{c['name']:<30} {c['directCount']:>5} {c['l2Count']:>5} {c['l3Count']:>5} {s:>6}")
print("-"*55)
print(f"{'RAZEM':<30} {'':>5} {'':>5} {'':>5} {total:>6}")

# Total all products
all_count = q('count(*[_type=="product" && !(name match "P-*")])')
placeholder_count = q('count(*[_type=="product" && name match "P-*"])')
print(f"\nWszystkie produkty (non-placeholder): {all_count}")
print(f"Placeholdery P-*: {placeholder_count}")


Kategoria                         L1    L2    L3    SUM
-------------------------------------------------------
Chemia budowlana                   0    70  2672   2742
Dachy                              0    46   916    962
Farby i rozpuszczalniki            0   461  3669   4130
Izolacje                           0   131  2145   2276
Narzędzia i mocowania              0   137  1614   1751
Pozostałe                          0     0     1      1
Płytki                             0     2   629    631
Stropy i ściany                    0     0  1057   1057
Sucha zabudowa                     0     1   855    856
Sufity podwieszane                 0     2     7      9
-------------------------------------------------------
RAZEM                                             14415



Wszystkie produkty (non-placeholder): 15746
Placeholdery P-*: 90


In [15]:

import json, re
from collections import Counter

# 1. Sprawdź reimport_scraped.jsonl dla gipsy
path = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/reimport_scraped.jsonl"
gipsy_products = []
with open(path) as f:
    for line in f:
        p = json.loads(line)
        cats = p.get("categoryPath", "")
        if "gip" in cats.lower() or "gładź" in cats.lower() or "gladz" in cats.lower():
            gipsy_products.append({"name": p["name"][:50], "category": cats[:80]})

print(f"Produkty 'gipsy' w reimport_scraped.jsonl: {len(gipsy_products)}")
for p in gipsy_products[:10]:
    print(f"  {p['name']} | {p['category']}")

# 2. Sprawdź głębokość kategorii w Sanity
import urllib.request, urllib.parse
TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

# Gdzie jest gipsy-i-gladzie w hierarchii?
gipsy = q('*[_type=="category" && slug.current=="gipsy-i-gladzie"][0]{ _id, name, "slug": slug.current, "parentSlug": parent->slug.current, "grandParentSlug": parent->parent->slug.current }')
print(f"\nKategoria gipsy-i-gladzie: {gipsy}")

# L4 produkty (te które nie są liczone)
l4 = q('count(*[_type=="product" && !(name match "P-*") && defined(category->parent->parent->parent._ref)])')
print(f"\nProdukty na poziomie L4 (pomijane): {l4}")

# Ile produktów nie ma dopasowania w top 3 levels  
l4_sample = q('*[_type=="product" && !(name match "P-*") && defined(category->parent->parent->parent._ref)][0..4]{ name, "catSlug": category->slug.current, "l2": category->parent->slug.current, "l3": category->parent->parent->slug.current, "l4": category->parent->parent->parent->slug.current }')
print(f"\nPrzykłady L4:")
for p in l4_sample:
    print(f"  {p['name'][:40]} → {p['catSlug']} → {p['l2']} → {p['l3']} → {p['l4']}")


AttributeError: 'list' object has no attribute 'lower'

In [19]:

import urllib.request, urllib.parse, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

# Gdzie jest gipsy-i-gladzie w hierarchii?
gipsy = q('*[_type=="category" && slug.current=="gipsy-i-gladzie"][0]{ _id, name, "slug": slug.current, "parentSlug": parent->slug.current, "grandParentSlug": parent->parent->slug.current }')
print(f"Kategoria gipsy-i-gladzie:")
print(f"  parent:       {gipsy.get('parentSlug')}")
print(f"  grandParent:  {gipsy.get('grandParentSlug')}")

# L4 produkty
l4 = q('count(*[_type=="product" && !(name match "P-*") && defined(category->parent->parent->parent._ref)])')
print(f"\nProdukty L4 (pomijane przez 3-poziomowy query): {l4}")

# Przykłady L4
l4s = q('*[_type=="product" && !(name match "P-*") && defined(category->parent->parent->parent._ref)][0..3]{ name, "catSlug": category->slug.current, "l2": category->parent->slug.current, "l3": category->parent->parent->slug.current, "l4": category->parent->parent->parent->slug.current }')
for p in l4s:
    print(f"  L4: {p['name'][:35]} | {p['catSlug']} → {p['l2']} → {p['l3']} → {p['l4']}")

# Ile produktów w reimport z "Gipsy" w ścieżce (z JSONL - categoryPath jest listą)
path = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/reimport_scraped.jsonl"
gipsy_count = 0
with open(path) as f:
    for line in f:
        p = json.loads(line)
        cats = p.get("categoryPath", [])
        if isinstance(cats, list):
            cats_str = " ".join(cats).lower()
        else:
            cats_str = str(cats).lower()
        if "gip" in cats_str or "gładź" in cats_str or "gladz" in cats_str:
            gipsy_count += 1
print(f"\nProdukty 'gipsy' w reimport_scraped.jsonl: {gipsy_count}")


Kategoria gipsy-i-gladzie:
  parent:       chemia-budowlana
  grandParent:  None



Produkty L4 (pomijane przez 3-poziomowy query): 1331


  L4: Wełna szklana Isover Super-Vent Plu | weny-do-izolacji-fasad-wentylowanych → weny-fasadowe → weny → izolacje
  L4: Wełna skalna Isover Fasoterm 35 035 | weny-do-izolacji-fasad-niewentylowanych → weny-fasadowe → weny → izolacje
  L4: Wełna skalna Rockwool Ventirock L03 | weny-do-izolacji-fasad-wentylowanych → weny-fasadowe → weny → izolacje
  L4: Bloczek H+H Gold+ 4,0-500 PWU 175/2 | bloczki-beton-komorkowy → bloczki → materiay-konstrukcyjne → stropy-i-sciany



Produkty 'gipsy' w reimport_scraped.jsonl: 535


In [23]:

import urllib.request, urllib.parse, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

# Produkty z "gips" lub "głądź"/"gładź" w nazwie
gipsy_sanity = q('''*[_type=="product" && (name match "Gips*" || name match "*gips*" || name match "*Gładź*" || name match "*Gladz*" || name match "*gładź*") && !(name match "P-*")][0..9]{
  name,
  "catSlug": category->slug.current,
  "catName": category->name,
  "parentSlug": category->parent->slug.current,
  "parentName": category->parent->name
}''')

print(f"Przykłady produktów 'gips*' w Sanity ({len(gipsy_sanity)} pobranych):")
for p in gipsy_sanity:
    print(f"  {p['name'][:45]:45s} | kat: {p.get('catName','?')[:25]:25s} | parent: {p.get('parentName','?')[:20]}")

# Policz wszystkie
total_gipsy_sanity = q('''count(*[_type=="product" && (name match "Gips*" || name match "*gips*" || name match "*Gładź*" || name match "*Gladz*" || name match "*gładź*" || name match "Gład*") && !(name match "P-*")])''')
print(f"\nŁącznie produktów 'gips*'/'gładź*' w Sanity: {total_gipsy_sanity}")

# Rozkład po kategoriach
dist = q('''*[_type=="product" && (name match "Gips*" || name match "*gips*" || name match "*Gładź*" || name match "*Gladz*" || name match "*gładź*" || name match "Gład*") && !(name match "P-*")]{
  "catSlug": category->slug.current
}''')

from collections import Counter
cat_counts = Counter(p.get("catSlug","?") for p in dist)
print("\nRozkład po kategoriach:")
for cat, cnt in cat_counts.most_common(10):
    print(f"  {cat}: {cnt}")


Przykłady produktów 'gips*' w Sanity (10 pobranych):
  Tynk Gipsowy Knauf Goldband Gladki 25 Kg      | kat: Tynki gipsowe             | parent: Tynki
  Gładź cementowo-polimerowa Cekol C-35 biała 5 | kat: Gładzie gipsowe w proszku | parent: Gipsy i gładzie
  Gładź szpachlowa polimerowa ProGold 25kg      | kat: Gładzie masy gotowe       | parent: Gipsy i gładzie
  Gips szpachlowy Siniat Nida Max 5 kg          | kat: Gipsy szpachlowe          | parent: Gipsy i gładzie
  Płyta gipsowo-kartonowa Siniat Smart krawędź  | kat: Płyty gipsowo-kartonowe   | parent: Płyty
  Taśma bitumiczna Rigips 80 mm o dł, L 10 m    | kat: Hydroizolacje bitumiczne  | parent: Hydroizolacje
  Gips szpachlowy Siniat Nida Start 25 kg       | kat: Gipsy szpachlowe          | parent: Gipsy i gładzie
  Płyta gipsowo-kartonowa Siniat Nida Expert 12 | kat: Płyty gipsowo-kartonowe   | parent: Płyty
  Gładź szpachlowa Atlas Gipsar Uni wzmocniona  | kat: Gładzie gipsowe w proszku | parent: Gipsy i gładzie
  Super biała gł


Rozkład po kategoriach:
  wkrety-do-suchej-zabudowy: 112
  pyty-gipsowo-kartonowe: 96
  gadzie-masy-gotowe: 90
  gadzie-gipsowe-w-proszku: 61
  gipsy-szpachlowe: 58
  tynki-gipsowe: 35
  masy-szpachlowe-gotowe: 17
  aczniki-do-profili: 12
  tynki-cementowo-wapienne: 10
  tynki-specjalne: 8


In [27]:

import urllib.request, urllib.parse, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())["result"]

# Pobierz WSZYSTKIE kategorie 
all_cats = q('*[_type=="category"]{ _id, name, "slug": slug.current, "parentId": parent._ref }')
print(f"Wszystkich kategorii w Sanity: {len(all_cats)}")

# Rozkład slugów - które SLUG mają produkty?
# Pobierz wszystkie category->slug.current z produktów
prod_cat_slugs = q('*[_type=="product" && !(name match "P-*")]{ "catId": category._ref }')
cat_id_counts = {}
for p in prod_cat_slugs:
    cid = p.get("catId")
    if cid:
        cat_id_counts[cid] = cat_id_counts.get(cid, 0) + 1

# Mapa id→slug
id_to_cat = {c["_id"]: c for c in all_cats}

# Znajdź kategorie z produktami które mają "błędny" slug (ł usunięty zamiast l)
# Wzorzec: prawidłowy slug ma transliterację (ł→l, ą→a itp.), błędny slug ma dziury
# Heurystyka: slug zawiera ciąg "gl" lub "pl" lub "kl" itp. w poprawnej, a w błędnej "g" "p" "k" bez l

# Najpierw: wylistuj kategorie z produktami
cats_with_products = []
for cat_id, count in cat_id_counts.items():
    if cat_id in id_to_cat:
        c = id_to_cat[cat_id]
        cats_with_products.append({"id": cat_id, "slug": c["slug"], "name": c["name"], "count": count})

cats_with_products.sort(key=lambda x: -x["count"])
print(f"\nKategorie z produktami (top 20):")
for c in cats_with_products[:20]:
    print(f"  {c['slug']:<45} {c['count']:>5} prod  | {c['name']}")

print(f"\nKategorie z produktami łącznie: {len(cats_with_products)}")


Wszystkich kategorii w Sanity: 551



Kategorie z produktami (top 20):
  grunty-uniwersalne                              335 prod  | Grunty uniwersalne
  spoiny-elastyczne                               323 prod  | Spoiny elastyczne
  styropiany-fasadowe-eps                         312 prod  | Styropiany fasadowe EPS
  impregnaty                                      304 prod  | Impregnaty
  farby-wewnetrzne-kolorowe                       302 prod  | Farby wewnętrzne kolorowe
  okna-dachowe                                    300 prod  | Okna dachowe
  dachowki-ceramiczne                             300 prod  | Dachówki ceramiczne
  schody-strychowe                                300 prod  | Schody strychowe
  farby-elewacyjne-silikonowe                     299 prod  | Farby elewacyjne silikonowe
  farby-elewacyjne-silikatowe                     299 prod  | Farby elewacyjne silikatowe
  farby-elewacyjne-akrylowe                       299 prod  | Farby elewacyjne akrylowe
  pedzle                                          297 

In [31]:

import urllib.request, urllib.parse, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())["result"]

# Transliteracja PL→ASCII
def slugify(text):
    pl = {'ą':'a','ć':'c','ę':'e','ł':'l','ń':'n','ó':'o','ś':'s','ź':'z','ż':'z',
          'Ą':'a','Ć':'c','Ę':'e','Ł':'l','Ń':'n','Ó':'o','Ś':'s','Ź':'z','Ż':'z'}
    result = ""
    for ch in text.lower():
        result += pl.get(ch, ch)
    import re
    result = re.sub(r'[^a-z0-9]+', '-', result).strip('-')
    return result

# Pobierz WSZYSTKIE kategorie z produktami i ichnazwy
prod_cat = q('*[_type=="product" && !(name match "P-*")]{ "catId": category._ref, "catSlug": category->slug.current, "catName": category->name }')
print(f"Produktów: {len(prod_cat)}")

# Grupuj wg catId
from collections import defaultdict
by_cat_id = defaultdict(list)
for p in prod_cat:
    by_cat_id[p.get("catId")].append(p)

# Pobierz wszystkie kategorie
all_cats = q('*[_type=="category"]{ _id, name, "slug": slug.current, "parentId": parent._ref }')
id_to_cat = {c["_id"]: c for c in all_cats}
slug_to_cat = {c["slug"]: c for c in all_cats}

# Znajdź kategorie z produktami gdzie OBECNY slug != expectedSlug(name)
mismatches = []
products_affected = 0
for cat_id, prods in by_cat_id.items():
    cat = id_to_cat.get(cat_id)
    if not cat:
        continue
    current_slug = cat.get("slug", "")
    expected_slug = slugify(cat.get("name", ""))
    if current_slug != expected_slug:
        # Sprawdź czy istnieje kat z poprawnym slugiem
        correct_cat = slug_to_cat.get(expected_slug)
        mismatches.append({
            "old_id": cat_id,
            "old_slug": current_slug,
            "expected_slug": expected_slug,
            "name": cat.get("name"),
            "new_id": correct_cat["_id"] if correct_cat else None,
            "prod_count": len(prods)
        })
        products_affected += len(prods)

mismatches.sort(key=lambda x: -x["prod_count"])
print(f"\nKategorie z błędnym slugiem (produkty dostępne): {len(mismatches)}")
print(f"Produktów dotkniętych: {products_affected}")
print(f"\nTop 15 błędnych kategorii:")
for m in mismatches[:15]:
    has_correct = "✅ new_cat" if m["new_id"] else "❌ brak"
    print(f"  {m['old_slug']:<40} → {m['expected_slug']:<40} | {m['prod_count']:>4} | {has_correct}")


Produktów: 15746

Kategorie z błędnym slugiem (produkty dostępne): 48
Produktów dotkniętych: 3353

Top 15 błędnych kategorii:
  aczniki-do-izolacji-fasadowych           → laczniki-do-izolacji-fasadowych          |  295 | ✅ new_cat
  konierze-i-produkty-instalacyjne         → kolnierze-i-produkty-instalacyjne        |  269 | ❌ brak
  farby-wewnetrzne-biae                    → farby-wewnetrzne-biale                   |  264 | ✅ new_cat
  koki-i-wkrety-uniwersalne                → kolki-i-wkrety-uniwersalne               |  264 | ✅ new_cat
  styropian-dach-podoga-eps                → styropian-dach-podloga-eps               |  262 | ✅ new_cat
  pytki-scienne                            → plytki-scienne                           |  206 | ✅ new_cat
  pytki-tarasowe                           → plytki-tarasowe                          |  189 | ✅ new_cat
  weny-do-poddaszy                         → welny-do-poddaszy                        |  142 | ✅ new_cat
  aczniki-do-profili                 

In [35]:

import json

# mismatches jest z poprzedniej komórki — piszemy plik remap
remap_path = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/cat_remap.json"

# Zbuduj remap: old_id → new_id (tylko te gdzie new_id istnieje)
remap = {}
no_match = []

for m in mismatches:
    if m["new_id"]:
        remap[m["old_id"]] = {
            "new_id": m["new_id"],
            "old_slug": m["old_slug"],
            "new_slug": m["expected_slug"],
            "name": m["name"],
            "prod_count": m["prod_count"]
        }
    else:
        no_match.append(m)

with open(remap_path, "w") as f:
    json.dump({"remap": remap, "no_match": no_match}, f, ensure_ascii=False, indent=2)

print(f"Remap zapisany: {len(remap)} kategorii do przepięcia")
print(f"Brak odpowiednika: {len(no_match)} kategorii (będą naprawione przez zmianę slug)")
print(f"\nBez odpowiednika:")
for m in no_match:
    print(f"  {m['old_slug']} → {m['expected_slug']} ({m['prod_count']} prod)")

total_remappable = sum(v["prod_count"] for v in remap.values())
print(f"\nProduktów do przepięcia (old→new): {total_remappable}")


Remap zapisany: 37 kategorii do przepięcia
Brak odpowiednika: 11 kategorii (będą naprawione przez zmianę slug)

Bez odpowiednika:
  konierze-i-produkty-instalacyjne → kolnierze-i-produkty-instalacyjne (269 prod)
  pyty-elewacyjne-pir → plyty-elewacyjne-pir (127 prod)
  weny-do-izolacji-fasad-niewentylowanych → welny-do-izolacji-fasad-niewentylowanych (86 prod)
  weny-do-izolacji-fasad-wentylowanych → welny-do-izolacji-fasad-wentylowanych (71 prod)
  akcesoria-do-farb-przemysowych → akcesoria-do-farb-przemyslowych (22 prod)
  pyty-elewacyjne-pur → plyty-elewacyjne-pur (8 prod)
  lakiery-pozostae → lakiery-pozostale (2 prod)
  farby-elewacyjne-pozostae → farby-elewacyjne-pozostale (2 prod)
  folie-w-pynie → folie-w-plynie (1 prod)
  pozostae-narzedzia-reczne → pozostale-narzedzia-reczne (1 prod)
  pyty-dachowe-pir → plyty-dachowe-pir (1 prod)

Produktów do przepięcia (old→new): 2763


In [4]:

import urllib.request, urllib.parse, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

prod = q('*[_type=="product" && slug.current=="plyta-sufitowa-kcs-armstrong-perla-board-600x600x15-mm"][0]{ _id, name, sku, "images": images[].asset->url, "catName": category->name }')
print("ID:", prod.get("_id"))
print("SKU:", prod.get("sku"))
print("Nazwa:", prod.get("name"))
print("Kat:", prod.get("catName"))
print("Zdjęcia:")
for url in (prod.get("images") or []):
    print(" ", url)


ID: product-p137
SKU: ARM-DUNE-600
Nazwa: Plyta Sufitowa Kcs Armstrong Perla Board 600X600X15 Mm
Kat: Płyty sufitowe z wełny mineralnej
Zdjęcia:
  https://cdn.sanity.io/images/nzcwegq7/production/4660f472c21946386a0633db3cb1e2af7bd1b85d-1200x1200.jpg


In [8]:

import json, urllib.request, urllib.parse

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"
BAD_ASSET = "4660f472c21946386a0633db3cb1e2af7bd1b85d"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

# 1. Ile produktów w Sanity ma to samo złe zdjęcie?
bad_products = q(f'*[_type=="product" && "{BAD_ASSET}" in images[].asset->_id]{{ _id, name, sku }}')
print(f"Produkty z tym samym błędnym zdjęciem ({BAD_ASSET[:12]}...):")
for p in bad_products:
    print(f"  {p['_id']} | {p['sku']} | {p['name'][:50]}")
print(f"Łącznie: {len(bad_products)}")

# 2. Szukaj poprawnego URL w JSONL po SKU/nazwie
JSONL = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/reimport_scraped.jsonl"
search_sku = "ARM-DUNE-600"
search_name = "armstrong perla"

found = []
with open(JSONL) as f:
    for line in f:
        p = json.loads(line)
        sku = (p.get("sku") or "").upper()
        name = (p.get("name") or "").lower()
        if search_sku in sku or search_name in name or "perla board" in name or "kcs armstrong" in name:
            found.append({"sku": p.get("sku"), "name": p.get("name"), "imageUrl": p.get("imageUrl")})

print(f"\nDopasowania w JSONL:")
for f_ in found:
    print(f"  SKU: {f_['sku']} | {f_['name'][:55]} | img: {f_['imageUrl']}")


Produkty z tym samym błędnym zdjęciem (4660f472c219...):
Łącznie: 0



Dopasowania w JSONL:


In [12]:

import json

WORKDIR = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud"

# 1. Sprawdź img_to_fix_log.json
with open(f"{WORKDIR}/img_to_fix_log.json") as f:
    fix_log = json.load(f)

p137 = fix_log.get("product-p137")
print("img_to_fix_log product-p137:", p137)

# 2. Sprawdź updated_products.json
with open(f"{WORKDIR}/updated_products.json") as f:
    updated = json.load(f)

upd = next((x for x in updated if x.get("productId") == "product-p137" or x.get("_id") == "product-p137"), None)
print("updated_products.json product-p137:", upd)

# 3. Sprawdź revert_log.json
with open(f"{WORKDIR}/revert_log.json") as f:
    revert = json.load(f)

rev = next((x for x in revert if x.get("productId") == "product-p137" or x.get("id") == "product-p137"), None)
print("revert_log product-p137:", rev)

# 4. Szukaj w JSONL po ID p137 / P-137
with open(f"{WORKDIR}/reimport_scraped.jsonl") as f:
    for i, line in enumerate(f):
        p = json.loads(line)
        sku = str(p.get("sku") or "")
        iurl = str(p.get("imageUrl") or "")
        if "P-137" in sku or "p-137" in sku.lower() or "137" == sku:
            print(f"JSONL[{i}] SKU={sku} name={p.get('name','')[:50]} img={iurl[:60]}")


AttributeError: 'list' object has no attribute 'get'

In [16]:

import json

WORKDIR = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud"

# img_to_fix_log — lista
with open(f"{WORKDIR}/img_to_fix_log.json") as f:
    fix_log = json.load(f)

p137_fix = [x for x in fix_log if x.get("productId") == "product-p137"]
print("img_to_fix_log:", p137_fix)

# updated_products — lista
with open(f"{WORKDIR}/updated_products.json") as f:
    updated = json.load(f)

p137_upd = [x for x in updated if x.get("productId") == "product-p137"]
print("updated_products:", p137_upd)

# revert_log — lista
with open(f"{WORKDIR}/revert_log.json") as f:
    revert = json.load(f)

p137_rev = [x for x in revert if x.get("productId") == "product-p137"]
print("revert_log:", p137_rev)

# JSONL — szukaj po P-137 i armstrong/sufitowa
with open(f"{WORKDIR}/reimport_scraped.jsonl") as f:
    for i, line in enumerate(f):
        p = json.loads(line)
        sku = str(p.get("sku") or "").upper()
        name = str(p.get("name") or "").lower()
        if "P-137" == sku or "armstrong" in name and "sufitow" in name:
            print(f"JSONL[{i}] sku={p.get('sku')} | {p.get('name','')[:55]} | img={p.get('imageUrl','')[:70]}")


img_to_fix_log: []
updated_products: []
revert_log: []


In [20]:

import json, urllib.request

WORKDIR = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud"

# 1. Armstrong ceiling tiles w JSONL
armstrong = []
with open(f"{WORKDIR}/reimport_scraped.jsonl") as f:
    for line in f:
        p = json.loads(line)
        name = (p.get("name") or "").lower()
        if "armstrong" in name:
            armstrong.append({"sku": p.get("sku"), "name": p.get("name","")[:60], "img": p.get("imageUrl","")[:80]})

print(f"Armstrong w JSONL: {len(armstrong)}")
for a in armstrong[:10]:
    print(f"  {a['sku']:12} | {a['name']:<55} | {a['img'][:50]}")

# 2. Spróbuj bechcicki.pl P-137
url = "https://www.bechcicki.pl/P-137-id-p-P-137"
try:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=10) as r:
        html = r.read().decode("utf-8", errors="ignore")
    # Szukaj URL zdjęcia
    import re
    imgs = re.findall(r'https?://[^\s"\']+\.(?:jpg|webp|png)[^\s"\']*', html)
    product_imgs = [i for i in imgs if "product" in i.lower() or "upload" in i.lower() or "media" in i.lower()]
    print(f"\nbechcicki.pl P-137 HTTP OK, zdjęcia: {product_imgs[:3]}")
except Exception as e:
    print(f"\nbechcicki.pl P-137: {e}")


Armstrong w JSONL: 0



bechcicki.pl P-137: HTTP Error 500: Internal Server Error


In [24]:

import urllib.request, json, io, re

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"
DATASET = "production"

# Kandydaci na poprawne zdjęcie (w kolejności preferencji)
IMAGE_CANDIDATES = [
    "https://www.artbud.pl/pl/p/img/14308/armstrong-perla.jpg",
    "https://sklep.sufitysystemowe.pl/wp-content/uploads/2021/01/Armstrong-Perla.jpg",
    "https://sc04.alicdn.com/kf/H68b1d90803cc4d889cae8b892be87ce89.jpg",
]

def try_download(url):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})
        with urllib.request.urlopen(req, timeout=15) as r:
            data = r.read()
            ct = r.headers.get("Content-Type", "image/jpeg")
            if len(data) > 5000 and ("image" in ct or url.endswith((".jpg",".webp",".png"))):
                return data, ct
    except Exception as e:
        print(f"  ✗ {url[:60]} → {e}")
    return None, None

img_data = None
img_ct = None
for url in IMAGE_CANDIDATES:
    print(f"Próba: {url[:70]}")
    img_data, img_ct = try_download(url)
    if img_data:
        print(f"  ✓ Pobrano {len(img_data)//1024} kB  ct={img_ct}")
        break

if not img_data:
    # Ostatnia deska ratunku: knauf.com
    url = "https://media.knauf.com/a/6vp1rQpP56KKiX9vdSFjbg"
    print(f"Próba knauf: {url}")
    img_data, img_ct = try_download(url)
    if img_data:
        print(f"  ✓ Pobrano {len(img_data)//1024} kB")

if not img_data:
    print("❌ Nie udało się pobrać żadnego zdjęcia.")
else:
    # Upload do Sanity Assets API
    ext = "jpg" if "jpeg" in (img_ct or "") or "jpg" in (img_ct or "") else "webp"
    upload_url = f"https://{PROJECT}.api.sanity.io/v2023-08-01/assets/images/{DATASET}"
    req = urllib.request.Request(
        upload_url,
        data=img_data,
        headers={
            "Authorization": f"Bearer {TOKEN}",
            "Content-Type": img_ct or "image/jpeg",
        }
    )
    with urllib.request.urlopen(req, timeout=30) as r:
        asset_resp = json.loads(r.read())

    asset_id = asset_resp["document"]["_id"]
    asset_url = asset_resp["document"]["url"]
    print(f"\n✅ Asset wgrany do Sanity:")
    print(f"   _id: {asset_id}")
    print(f"   url: {asset_url}")

    # Patch product-p137 → nowe zdjęcie
    mutation = {
        "mutations": [{
            "patch": {
                "id": "product-p137",
                "set": {
                    "images": [{"_type": "image", "_key": "main", "asset": {"_type": "reference", "_ref": asset_id}}]
                }
            }
        }]
    }
    mutate_url = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"
    req2 = urllib.request.Request(
        mutate_url,
        data=json.dumps(mutation).encode(),
        headers={"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req2, timeout=30) as r2:
        result = json.loads(r2.read())
    print(f"\n✅ product-p137 zaktualizowany: {result.get('results', [{}])[0]}")


Próba: https://www.artbud.pl/pl/p/img/14308/armstrong-perla.jpg


  ✗ https://www.artbud.pl/pl/p/img/14308/armstrong-perla.jpg → HTTP Error 404: Not Found
Próba: https://sklep.sufitysystemowe.pl/wp-content/uploads/2021/01/Armstrong-


  ✗ https://sklep.sufitysystemowe.pl/wp-content/uploads/2021/01/ → HTTP Error 404: Not Found
Próba: https://sc04.alicdn.com/kf/H68b1d90803cc4d889cae8b892be87ce89.jpg


  ✓ Pobrano 510 kB  ct=image/jpeg



✅ Asset wgrany do Sanity:
   _id: image-c5b3f9ef9dafa5ca09ff3c750bc6d2cdf54503ec-800x800-jpg
   url: https://cdn.sanity.io/images/nzcwegq7/production/c5b3f9ef9dafa5ca09ff3c750bc6d2cdf54503ec-800x800.jpg



✅ product-p137 zaktualizowany: {'operation': 'update'}


In [28]:

import urllib.request, urllib.parse, json
from collections import defaultdict

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())["result"]

# Pobierz produkty + ich pierwszy asset ID
# Jeśli ten sam asset _id pojawia się u wielu różnych produktów → podejrzane cross-przypisanie
print("Pobieranie produktów z asset ID...")
prods = q('''*[_type=="product" && defined(images[0]) && !(name match "P-*")][0...16000]{
  _id, name,
  "catSlug": category->slug.current,
  "assetId": images[0].asset._ref
}''')
print(f"Produktów z obrazkiem: {len(prods)}")

# Grupuj po assetId — te same zdjęcie u wielu produktów
by_asset = defaultdict(list)
for p in prods:
    aid = p.get("assetId")
    if aid:
        by_asset[aid].append({"id": p["_id"], "name": p.get("name","")[:45], "cat": p.get("catSlug","")})

# Znajdź asset ID używane przez wiele produktów (> 3)
duplicates = {aid: plist for aid, plist in by_asset.items() if len(plist) > 3}
print(f"\nAsset ID użyte przez >3 różnych produktów: {len(duplicates)}")
total_affected = sum(len(v) for v in duplicates.values())
print(f"Produktów dotkniętych: {total_affected}")

# Pokaż top 10 najgorszych
sorted_dups = sorted(duplicates.items(), key=lambda x: -len(x[1]))
print("\nTop duplikaty (potencjalnie złe przypisania):")
for aid, plist in sorted_dups[:8]:
    cats = set(p["cat"] for p in plist)
    print(f"\n  Asset: ...{aid[-20:]} → {len(plist)} produktów, {len(cats)} kategorii")
    for p in plist[:3]:
        print(f"    {p['id']:15} | {p['cat']:30} | {p['name']}")
    if len(plist) > 3:
        print(f"    ... i {len(plist)-3} więcej")


Pobieranie produktów z asset ID...


Produktów z obrazkiem: 14737

Asset ID użyte przez >3 różnych produktów: 634
Produktów dotkniętych: 7248

Top duplikaty (potencjalnie złe przypisania):

  Asset: ...f4c102-1200x1200-jpg → 276 produktów, 1 kategorii
    product-p0013490 | farby-elewacyjne-silikatowe    | Farba silikatowa Baumit SilikatColor do elewa
    product-p0050671 | farby-elewacyjne-silikatowe    | Farba silikatowa Baumit SilikatColor do elewa
    product-p0161282 | farby-elewacyjne-silikatowe    | Farba silikatowa Baumit SilikatColor do elewa
    ... i 273 więcej

  Asset: ...bdaa95-1200x1200-jpg → 160 produktów, 1 kategorii
    product-p0257356 | farby-elewacyjne-akrylowe      | Farba akrylowa Baumit PuraColor do elewacji 0
    product-p0257365 | farby-elewacyjne-akrylowe      | Farba akrylowa Baumit PuraColor do elewacji 0
    product-p0257375 | farby-elewacyjne-akrylowe      | Farba akrylowa Baumit PuraColor do elewacji 0
    ... i 157 więcej

  Asset: ...bbdc40-1200x1200-jpg → 159 produktów, 1 kategorii
    p

In [32]:

import urllib.request, urllib.parse, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"
# Asset ID acetonu Dorex (pełny format Sanity)
BAD_ASSET_REF = "image-4660f472c21946386a0633db3cb1e2af7bd1b85d-1200x1200-jpg"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())["result"]

bad = q(f'''*[_type=="product" && images[0].asset._ref == "{BAD_ASSET_REF}"]{{
  _id, name, sku,
  "catSlug": category->slug.current,
  "catName": category->name
}}''')

print(f"Produkty z Dorex-assetem: {len(bad)}")

# Załaduj JSONL do słownika SKU→imageUrl
JSONL = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/reimport_scraped.jsonl"
sku_to_img = {}
id_to_img  = {}
with open(JSONL) as f:
    for line in f:
        p = json.loads(line)
        sku = (p.get("sku") or "").strip().upper()
        pid = (p.get("sku") or "").strip()      # bechcicki P-XXXXXXX
        img = p.get("imageUrl") or ""
        if sku and img:
            sku_to_img[sku] = img
        if pid and img:
            id_to_img[pid] = img

# Dopasuj każdy produkt
matched, unmatched = [], []
for p in bad:
    prod_id = p["_id"]  # np. product-p028
    sku     = (p.get("sku") or "").strip().upper()
    # Spróbuj przez SKU
    img = sku_to_img.get(sku)
    # Spróbuj przez bechcicki ID (product-p028 → P-028)
    if not img:
        bech_id = prod_id.replace("product-", "").upper()  # P028
        img = id_to_img.get(bech_id) or id_to_img.get(f"P-{bech_id.lstrip('P')}")
    if img:
        matched.append({**p, "correctImg": img})
    else:
        unmatched.append(p)

print(f"Dopasowane (mają URL z JSONL): {len(matched)}")
print(f"Bez dopasowania:               {len(unmatched)}")
print("\nPrzykłady dopasowanych:")
for p in matched[:5]:
    print(f"  {p['_id']:18} | {p['sku']:15} | {p['correctImg'][:60]}")
print("\nBez dopasowania (top 10):")
for p in unmatched[:10]:
    print(f"  {p['_id']:18} | {p['sku']:15} | {p['catName']}")

# Zapisz listy do pliku
WORKDIR = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud"
with open(f"{WORKDIR}/bad_img_matched.json", "w") as f:
    json.dump(matched, f, ensure_ascii=False, indent=2)
with open(f"{WORKDIR}/bad_img_unmatched.json", "w") as f:
    json.dump(unmatched, f, ensure_ascii=False, indent=2)
print(f"\nZapisano: bad_img_matched.json ({len(matched)}) / bad_img_unmatched.json ({len(unmatched)})")


Produkty z Dorex-assetem: 127


Dopasowane (mają URL z JSONL): 1
Bez dopasowania:               126

Przykłady dopasowanych:
  product-p0196018   | P-0196018       | https://static.www.bechcicki.pl/P-/01/96/01/8/1/BIG.webp

Bez dopasowania (top 10):
  product-p028       | FCH-UX860-100   | Kołki i wkręty uniwersalne
  product-p029       | DRK-DVN-150     | Folie paroprzepuszczalne
  product-p030       | KTP-JZY-3       | Gonty bitumiczne
  product-p031       | BBT-HOL-6-SZ    | Kostka brukowa
  product-p032       | KNF-BD25-25     | Zaprawy posadzkowe masy samopoziomujące
  product-p034       | BAU-MPI25-25    | Tynki cementowo-wapienne
  product-p035       | ISV-AKU-50      | Wełny fasadowe
  product-p036       | GAM-R125-3-SZ   | Systemy rynnowe PVC
  product-p037       | ICP-V60S40-10   | Papy hydroizolacyjne
  product-p038       | LDE-JUL-CZW     | Płytki elewacyjne

Zapisano: bad_img_matched.json (1) / bad_img_unmatched.json (126)


In [36]:

import urllib.request, urllib.parse, json, time

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT  = "nzcwegq7"
DATASET  = "production"
BASE_Q   = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
BASE_M   = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"
BAD_REF  = "image-4660f472c21946386a0633db3cb1e2af7bd1b85d-1200x1200-jpg"

def q(groq):
    url = BASE_Q + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())["result"]

def mutate(mutations):
    data = json.dumps({"mutations": mutations}).encode()
    req  = urllib.request.Request(BASE_M, data=data, headers={
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json"
    })
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())

# Pobierz pełne tablice images dla tych produktów
bad_prods = q(f'''*[_type=="product" && images[0].asset._ref == "{BAD_REF}"]{{
  _id, name, sku,
  "allImages": images[]{{ _key, _type, "ref": asset._ref }}
}}''')

only_bad  = [p for p in bad_prods if len(p.get("allImages") or []) <= 1]
has_other = [p for p in bad_prods if len(p.get("allImages") or []) > 1]
print(f"Produktów z TYLKO złym zdjęciem: {len(only_bad)}")
print(f"Produktów z innymi zdjęciami też: {len(has_other)}")

if has_other:
    print("\nPrzykłady z dodatkowymi zdjęciami:")
    for p in has_other[:3]:
        print(f"  {p['_id']} | {p['name'][:40]} | images: {len(p['allImages'])}")

# ── FIX: usuń zły asset ze wszystkich produktów ──────────────────────────────
print(f"\nNaprawiam {len(bad_prods)} produktów...")

# Dla produktów z TYLKO złym zdjęciem → images = []
# Dla produktów z dodatkowymi → odfiltruj zły ref
patched = 0
errors  = 0
CHUNK   = 30

for i in range(0, len(bad_prods), CHUNK):
    chunk    = bad_prods[i:i+CHUNK]
    mutations = []
    for p in chunk:
        imgs = p.get("allImages") or []
        clean_imgs = [img for img in imgs if img.get("ref") != BAD_REF]
        mutations.append({
            "patch": {"id": p["_id"], "set": {"images": clean_imgs}}
        })
    try:
        r = mutate(mutations)
        patched += len(r.get("results", []))
    except Exception as e:
        errors += len(chunk)
        print(f"  ❌ chunk {i//CHUNK+1}: {e}")
    time.sleep(0.3)

print(f"\n✅ Zaktualizowano: {patched}")
print(f"❌ Błędy:         {errors}")

# Zapisz listę do późniejszego uzupełnienia zdjęć
WORKDIR = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud"
with open(f"{WORKDIR}/no_img_fix_needed.json", "w") as f:
    payload = [{"_id": p["_id"], "name": p["name"], "sku": p.get("sku","")} for p in bad_prods]
    json.dump(payload, f, ensure_ascii=False, indent=2)
print(f"Lista produktów bez zdjęcia zapisana: no_img_fix_needed.json")


Produktów z TYLKO złym zdjęciem: 127
Produktów z innymi zdjęciami też: 0

Naprawiam 127 produktów...



✅ Zaktualizowano: 127
❌ Błędy:         0
Lista produktów bez zdjęcia zapisana: no_img_fix_needed.json


In [4]:

import urllib.request, urllib.parse, json
from PIL import Image
import io, numpy as np

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"

def q(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

# Pobierz dane produktu
p = q('*[_type=="product" && slug.current=="styropian-fasadowy-swisspor-eps-70-031-15cm"][0]{ _id, name, sku, "imgUrl": images[0].asset->url }')
print(f"ID: {p['_id']}")
print(f"Img: {p['imgUrl']}")

# Pobierz i przeanalizuj obraz (thumbnail 200x200)
thumb_url = p['imgUrl'] + "?w=400&h=400&fit=crop&auto=format"
req = urllib.request.Request(thumb_url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=20) as r:
    img_data = r.read()

img = Image.open(io.BytesIO(img_data)).convert("RGB")
arr = np.array(img)
h, w = arr.shape[:2]
print(f"Rozmiar: {w}x{h}")

# Analiza pikseli — ile ciemnych (<40) w różnych strefach
dark_thresh = 40
dark = (arr.max(axis=2) < dark_thresh)

# Strefa wewnętrzna (20%-80%)
margin_h, margin_w = int(h*0.20), int(w*0.20)
inner = dark[margin_h:h-margin_h, margin_w:w-margin_w]
border_mask = np.ones_like(dark, dtype=bool)
border_mask[margin_h:h-margin_h, margin_w:w-margin_w] = False
border = dark[border_mask]

inner_ratio = inner.sum() / inner.size
border_ratio = border.sum() / border.size
total_dark = dark.sum() / dark.size

print(f"\nAnaliza pikseli:")
print(f"  Ogółem ciemnych (<40): {total_dark:.1%}")
print(f"  Strefa wewnętrzna (20-80%): {inner_ratio:.1%}")
print(f"  Obwódka (0-20%): {border_ratio:.1%}")
print(f"\n  -> Damage ratio inner: {inner_ratio:.4f}")

# Znajdź bechcicki URL dla tego produktu
JSONL = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/reimport_scraped.jsonl"
sku = p.get("sku", "")
bech_url = None
with open(JSONL) as f:
    for line in f:
        item = json.loads(line)
        if (item.get("sku") or "").upper() == sku.upper():
            bech_url = item.get("imageUrl")
            break
print(f"\nBechcicki URL dla {sku}: {bech_url}")


ID: prod-styropian-swisspor-eps70
Img: https://cdn.sanity.io/images/nzcwegq7/production/b4df9ba85c5a974830a2fdc2993cc69b5ee07aa7-1200x1200.jpg


Rozmiar: 400x400

Analiza pikseli:
  Ogółem ciemnych (<40): 92.7%
  Strefa wewnętrzna (20-80%): 81.0%
  Obwódka (0-20%): 99.2%

  -> Damage ratio inner: 0.8103



Bechcicki URL dla SWP-EPS70-031-15: None


In [8]:

import json

WORKDIR = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud"

with open(f"{WORKDIR}/updated_products.json") as f:
    updated = json.load(f)

# Struktura
print("Struktura pierwszego rekordu:", list(updated[0].keys()))
print("Przykład:", updated[0])
print(f"\nŁącznie: {len(updated)}")

# Szukaj styropianu
styro = [x for x in updated if "styropian-swisspor-eps70" in str(x).lower() or "swisspor" in str(x).lower() or "SWP-EPS70" in str(x)]
print(f"\nStyropian Swisspor w updated_products: {len(styro)}")
if styro:
    print(styro[0])


Struktura pierwszego rekordu: ['_id', 'url', 'status']
Przykład: {'_id': 'product-p0000015', 'url': 'https://cdn.sanity.io/images/nzcwegq7/production/bf71ad102869390c802d48f550cdb8764e20c607-700x700.webp', 'status': 'updated(130KB)'}

Łącznie: 9642

Styropian Swisspor w updated_products: 0


In [12]:

import json, urllib.request, urllib.parse, time

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE_M = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/mutate/production"
BASE_A = "https://nzcwegq7.api.sanity.io/v2023-08-01/assets/images/production"
WORKDIR = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud"

def mutate(mutations):
    data = json.dumps({"mutations": mutations}).encode()
    req = urllib.request.Request(BASE_M, data=data,
        headers={"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())

# === 1. NAPRAW product-p0196018 (Aceton Dorex — ma URL w JSONL) ===
print("=== Naprawiam Dorex Aceton p0196018 ===")
JSONL = f"{WORKDIR}/reimport_scraped.jsonl"
dorex_url = None
with open(JSONL) as f:
    for line in f:
        p = json.loads(line)
        if p.get("sku") == "P-0196018":
            dorex_url = p.get("imageUrl")
            break

if dorex_url:
    # Upload
    req = urllib.request.Request(dorex_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=20) as r:
        img_data = r.read()
    req2 = urllib.request.Request(BASE_A, data=img_data,
        headers={"Authorization": f"Bearer {TOKEN}", "Content-Type": "image/webp"})
    with urllib.request.urlopen(req2, timeout=30) as r2:
        asset = json.loads(r2.read())
    asset_id = asset["document"]["_id"]
    mutate([{"patch": {"id": "product-p0196018", "set": {
        "images": [{"_type": "image", "_key": "main", "asset": {"_type": "reference", "_ref": asset_id}}]
    }}}])
    print(f"✅ Dorex naprawiony → {asset_id}")
else:
    print("❌ Brak URL w JSONL dla Dorex")

# === 2. USUŃ 126 produktów bez zdjęć ===
print("\n=== Usuwam 126 produktów bez zdjęć ===")
with open(f"{WORKDIR}/no_img_fix_needed.json") as f:
    no_img = json.load(f)

# Wyklucz p0196018 (właśnie naprawiony)
to_delete = [p for p in no_img if p["_id"] != "product-p0196018"]
print(f"Do usunięcia: {len(to_delete)}")

deleted = 0
errors = 0
CHUNK = 50
for i in range(0, len(to_delete), CHUNK):
    chunk = to_delete[i:i+CHUNK]
    mutations = [{"delete": {"id": p["_id"]}} for p in chunk]
    try:
        r = mutate(mutations)
        deleted += len(r.get("results", []))
    except Exception as e:
        errors += len(chunk)
        print(f"  ❌ chunk error: {e}")
    time.sleep(0.3)

print(f"✅ Usuniętych: {deleted}")
print(f"❌ Błędy: {errors}")


=== Naprawiam Dorex Aceton p0196018 ===


✅ Dorex naprawiony → image-50b437f254ca09397b817ee05d6c51eea04351b8-1200x1200-webp

=== Usuwam 126 produktów bez zdjęć ===


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [16]:

import json, urllib.request, urllib.parse, time, os

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE_M = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/mutate/production"
BASE_Q = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"
WORKDIR = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud"

def mutate(mutations):
    data = json.dumps({"mutations": mutations}).encode()
    req = urllib.request.Request(BASE_M, data=data,
        headers={"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())

def q(groq):
    url = BASE_Q + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())["result"]

# Sprawdź plik
fpath = f"{WORKDIR}/no_img_fix_needed.json"
print(f"Plik istnieje: {os.path.exists(fpath)}, rozmiar: {os.path.getsize(fpath) if os.path.exists(fpath) else 0}")

if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
    with open(fpath) as f:
        no_img = json.load(f)
    print(f"Produktów w pliku: {len(no_img)}")
    to_delete = [p for p in no_img if p["_id"] != "product-p0196018"]
else:
    # Pobierz z Sanity — produkty bez zdjęć z krótkimi ID (p028-p160)
    print("Plik pusty — pobieram listę z Sanity...")
    to_delete_raw = q('''*[_type=="product" && !defined(images[0]) && _id match "product-p0*" && length(_id) <= 14]{
      _id, name, sku
    }''')
    # product-p028 ma 11 znaków, product-p0196018 ma 18 znaków
    to_delete = [p for p in to_delete_raw if p["_id"] != "product-p0196018" and len(p["_id"]) <= 14]
    print(f"Znaleziono z Sanity: {len(to_delete)}")
    if to_delete:
        print("Przykłady:", [(p["_id"], p.get("sku","")[:15]) for p in to_delete[:5]])

print(f"\nDo usunięcia: {len(to_delete)} produktów")

deleted = 0
CHUNK = 50
for i in range(0, len(to_delete), CHUNK):
    chunk = to_delete[i:i+CHUNK]
    mutations = [{"delete": {"id": p["_id"]}} for p in chunk]
    try:
        r = mutate(mutations)
        deleted += len(r.get("results", []))
    except Exception as e:
        print(f"  ❌ błąd chunk {i}: {e}")
    time.sleep(0.3)

print(f"\n✅ Usuniętych: {deleted}")


Plik istnieje: True, rozmiar: 0
Plik pusty — pobieram listę z Sanity...


Znaleziono z Sanity: 71
Przykłady: [('product-p008', 'CAP-SIL-10'), ('product-p013', 'BAU-MG5-25'), ('product-p014', 'GRZ-CEM1-425-25'), ('product-p017', 'DLX-JM-10W'), ('product-p018', 'CAP-AMP-10')]

Do usunięcia: 71 produktów



✅ Usuniętych: 71
